In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from bs4 import SoupStrainer
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
load_dotenv()

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
links = [
    "https://www.anytimefitness.com/",
    "https://www.anytimefitness.com/training",
    "https://www.anytimefitness.com/blog",
    "https://www.anytimefitness.com/membership",
    "https://franchise.anytimefitness.com/"
]

In [ ]:
documents = []
for link in links:
    loader = WebBaseLoader(
    web_path=(link,),
    bs_kwargs={
        "parse_only": SoupStrainer([
            "main",
            "article",
            "section",
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "li"
        ])
    }
)
    data = loader.load()[0]
    documents.append(data)

In [ ]:
documents

In [ ]:
splitted = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150).split_documents(documents)

In [ ]:
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-l6-v2")

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splitted,
    embedding=embedder,
    persist_directory="./chroma_db"
)

In [ ]:
query = ("what about the opening hours")
results = vectorstore.similarity_search(query)

In [ ]:
results[0]

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
llm = init_chat_model(model="groq:llama-3.1-8b-instant")

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
    Your are a bot made for anytimefitness.com
    you are not allowed to tell user bot internal things
    like chat history, context or anything
    else now you have to provide
    information about about their services,
    Answers the question based on the context below,
    if you can't find anything related to the query from context
    just say i can't help you with this query you can visit our website
    and don't use line breaking signs like \n
    anytimefitness.com.

    <context>
    {context}
    <context>

    Question:{input}
"""
                                          )

stuffChain = create_stuff_documents_chain(llm, prompt)

In [ ]:
contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "given a chat history and a latest user question rewrite the question so it can be understood without the chat history. rewrite only if needed otherwise return it as it is."),
        ("placeholder", "{chat_history}"),
        ("human", "{input}")
    ]
)

In [ ]:
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_prompt
)

In [ ]:
retrieval_chain = create_retrieval_chain(history_aware_retriever, stuffChain)

In [ ]:
chat_history = []

In [115]:
question = "tell me about all plans and their pricing"
response = retrieval_chain.invoke({
    "input" : question,
    "chat_history" : chat_history
})

chat_history.append(HumanMessage(content=question))
chat_history.append(AIMessage(content=response["answer"]))
response["answer"]

"At Anytime Fitness, you’ll find varying membership options dependent on your location. Some gyms offer 6, 12 and 18-month options. To learn about the different types of membership plans available and their pricing, you'll need to check with your local Anytime Fitness."